In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.DataFrame({'요일': ['월', '화', '수', '목', '금', '토', '일', '월', '수', '일']})
df

,요일
0,월
1,화
2,수
3,목
4,금
5,토
6,일
7,월
8,수
9,일


In [3]:
# 문자열을 정렬 가능한 범주형 데이터로 변환
weekdays = '월 화 수 목 금 토 일'.split()
df['요일'] = pd.Categorical(df['요일'], weekdays, ordered=True)


# weekdays: 요일의 “의도된 순서”를 담은 리스트 → ['월','화','수','목','금','토','일']
# df['요일']을 범주형(pd.Categorical)로 변환하고, 순서를 지정(ordered=True)
# 왜 why? -> 문자열 정렬은 사전순이라 ‘금’<‘목’ 같은 이상한 정렬이 나올 수 있으므로 범주형+순서를 주면 “월→일” 순으로 정렬/비교가 가능

In [4]:
# ‘요일’이 object가 아니라 categorical로 바뀌었는지 체크
df.dtypes 

요일    category
dtype: object

In [5]:
weekdays

['월', '화', '수', '목', '금', '토', '일']

데이터 유형
- 범주형   
명목형 데이터 순서나 우선순위가 없음 (색상)  
순서형 데이터 (초중고대, 요일, 영화등급)  
- 수치형   
이산형 특정값만 가질수있음   
연속형 연속된 값 (몸무게, 키, 나이, 소득)

In [6]:
# 범주형의 카테고리 목록 확인
df['요일'].cat.categories

Index(['월', '화', '수', '목', '금', '토', '일'], dtype='object')

In [7]:
# 범주형을 수치형으로 변환(월:0, 화:1, ..., 일:6)
df['요일'] = df['요일'].cat.codes

# .cat -> catregorical 관련 기능에 접근하기 위한 속성(attribute)
# .cat.codes -> 각 카테고리를 수치형으로 변환해주는 기능

In [8]:
df # ['월', '화', '수', '목', '금', '토', '일', '월', '수', '일']

,요일
0,0
1,1
2,2
3,3
4,4
5,5
6,6
7,0
8,2
9,6


In [9]:
# csv 파일 가져오기
df = pd.read_csv('data/test.csv')
df.head()

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   order_id            4622 non-null   int64 
 1   quantity            4622 non-null   int64 
 2   item_name           4622 non-null   object
 3   choice_description  3376 non-null   object
 4   item_price          4622 non-null   object
dtypes: int64(2), object(3)
memory usage: 180.7+ KB


In [11]:
# '$8.99' 같은 문자열 가격에서 $와 ,를 정규식으로 제거 후 실수형으로 변환 → '8.99'
df['item_price'] = df['item_price'].replace('[\$,]', '', regex=True).astype(float)
df.info()

# 정규식 [\$,] 의미: 대괄호 안의 문자 하나($ 또는 ,)를 찾겠다, regex=True로 정규식 사용 허용

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            4622 non-null   int64  
 1   quantity            4622 non-null   int64  
 2   item_name           4622 non-null   object 
 3   choice_description  3376 non-null   object 
 4   item_price          4622 non-null   float64
dtypes: float64(1), int64(2), object(2)
memory usage: 180.7+ KB


<>:2: SyntaxWarning: invalid escape sequence '\$'
<>:2: SyntaxWarning: invalid escape sequence '\$'
C:\Users\lg\AppData\Local\Temp\ipykernel_2844\2371791363.py:2: SyntaxWarning: invalid escape sequence '\$'
  df['item_price'] = df['item_price'].replace('[\$,]', '', regex=True).astype(float)


In [12]:
# 10달러 이상의 주문만 출력 >=, 조건 필터링의 기본 패턴을 보여주고, 전처리(문자→숫자)가 제대로 되었는지 확인하기 위함
df[df['item_price']>=10].head(1)

,order_id,quantity,item_name,choice_description,item_price
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98


In [13]:
# item_name == Chicken Bowl의 총 판매 수량 sum
df[df['item_name']=='Chicken Bowl']['quantity'].sum()

np.int64(761)

In [14]:
# choice_description 열의 결측치를 [] 으로 fillna -> choice_description NaN → '[]'로 채움 (빈 선택지를 의미)
df['choice_description'] = df['choice_description'].fillna('[]')
df.info()

# why? 이후 문자열 함수(.str.contains)는 NaN에서 에러/False를 만들 수 있으므로 먼저 결측 제거, 또한 모델/통계 전처리에서 NaN 제거는 필수

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            4622 non-null   int64  
 1   quantity            4622 non-null   int64  
 2   item_name           4622 non-null   object 
 3   choice_description  4622 non-null   object 
 4   item_price          4622 non-null   float64
dtypes: float64(1), int64(2), object(2)
memory usage: 180.7+ KB


In [15]:
'''
sum
count
std
var
median
cumsum 누적합
mode 최빈값
'''

'\nsum\ncount\nstd\nvar\nmedian\ncumsum 누적합\nmode 최빈값\n'

In [16]:
# item_name NaN → 최빈값(mode, 가장 많이 등장한 값)으로 채움
df['item_name'].fillna(df['item_name'].mode()[0], inplace=True)

C:\Users\lg\AppData\Local\Temp\ipykernel_2844\4257081981.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['item_name'].fillna(df['item_name'].mode()[0], inplace=True)


In [17]:
# choice_description에 'Black'이 포함된 행 필터링 후 상위 1개 출력
df.loc[df.choice_description.str.contains('Black')].head(1)

,order_id,quantity,item_name,choice_description,item_price
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98


In [18]:
# Vegetables 안 들어간 행의 개수, not(~)을 활용
len(df.loc[~df.choice_description.str.contains('Vegetables')])

# str.contains('Vegetables', na=False) # na=False 옵션: NaN을 False로 처리

3900

In [19]:
# item_name == Chicken Bowl  , quantity >= 2  and 조건 () & () 
df[ (df['item_name']=='Chicken Bowl') & (df['quantity'] >= 2)]

# 파이썬의 and가 아니라 판다스에서는 &(AND), |(OR), ~(NOT) 를 사용, 그리고 각 조건을 괄호로 감싸야 하는 점 주의!

,order_id,quantity,item_name,choice_description,item_price
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98
154,70,2,Chicken Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",17.50
282,124,2,Chicken Bowl,"[Fresh Tomato Salsa, [Rice, Black Beans, Chees...",17.50
409,178,3,Chicken Bowl,"[[Fresh Tomato Salsa (Mild), Tomatillo-Green C...",32.94
415,181,2,Chicken Bowl,[Tomatillo Red Chili Salsa],17.50
654,271,2,Chicken Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",17.50
976,401,2,Chicken Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",17.50
1017,418,2,Chicken Bowl,"[Fresh Tomato Salsa, [Rice, Cheese, Black Beans]]",17.50
1106,457,2,Chicken Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",17.50
1429,578,2,Chicken Bowl,"[Fresh Tomato Salsa, [Rice, Sour Cream, Guacam...",22.50


In [20]:

df = pd.read_csv('data/student_data.csv')
df.head()

,이름,나이,성별,지역,국어,영어,수학,과학,역사
0,김도윤,17.0,여,세종,54.0,NaN,55.0,79.0,61.0
1,송예나,19.0,남,강원,91.0,90.0,50.0,100.0,97.0
2,강하준,18.0,여,서울,99.0,78.0,95.0,99.0,79.0
3,김도윤,18.0,남,경기,55.0,96.0,52.0,78.0,50.0
4,송예나,19.0,여,울산,99.0,98.0,91.0,92.0,50.0


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   이름      95 non-null     object 
 1   나이      92 non-null     float64
 2   성별      95 non-null     object 
 3   지역      97 non-null     object 
 4   국어      96 non-null     float64
 5   영어      90 non-null     float64
 6   수학      99 non-null     float64
 7   과학      96 non-null     float64
 8   역사      95 non-null     float64
dtypes: float64(6), object(3)
memory usage: 7.2+ KB


In [22]:
# 수치형 데이터 컬럼만 파악하기
df.select_dtypes(include=np.number).columns

Index(['나이', '국어', '영어', '수학', '과학', '역사'], dtype='object')

In [23]:
# 안전하게 데이터를 씹고 뜯고 맛보고 즐기기 위해서 원본 df와 분리하여 df2에서 전처리 가공 진행
df2 = df.copy()

In [24]:
# 국어 결측치를 국어 평균값 mean으로 대체
df2['국어'] = df2['국어'].fillna(df2['국어'].mean())

In [25]:
# 수학 결측치를 수학 최빈값 mode로 대체 
df2['수학'] = df2['수학'].fillna(df2['수학'].mode()[0])

In [26]:
# 결측치 확인
df2.isnull().sum()

이름     5
나이     8
성별     5
지역     3
국어     0
영어    10
수학     0
과학     4
역사     5
dtype: int64

In [27]:
# 과학은 평균값, 영어는 최빈값으로 한꺼번에 대체, 딕셔너리 활용
df2.fillna(
    {
        '과학':df2['과학'].mean(),
        '영어':df2['영어'].mode()[0]
    },
    inplace=True
)
df2.isnull().sum()

이름    5
나이    8
성별    5
지역    3
국어    0
영어    0
수학    0
과학    0
역사    5
dtype: int64

In [28]:
# 목적에 따라 분석에 쓰지 않을 열 제거(차원 축소, 정보 배제 등)
df2 = df2.drop(columns=['성별'])

In [29]:
# 열 삭제 후, 결측이 남았는지 재확인
df2.isnull().sum()

이름    5
나이    8
지역    3
국어    0
영어    0
수학    0
과학    0
역사    5
dtype: int64

In [30]:
# NaN이 있는 행은 사용하지 않겠다 선언
df2 = df2.dropna() # 디폴트 값은 axis=0 행 제거, axis=1 열 제거는 df2.dropna(axis=1)으로 명시

In [31]:
df2.isnull().sum()

이름    0
나이    0
지역    0
국어    0
영어    0
수학    0
과학    0
역사    0
dtype: int64

#### SimpleImputer
- SimpleImputer는 사이킷런(sklearn)의 결측치(NaN, None) 자동 대체 도구
- SimpleImputer는 데이터셋 안의 결측값을 평균, 중앙값, 최빈값, 혹은 지정한 상수값으로 자동으로 채워주는 클래스
- impute → "결측치 처리" 관련 모듈, SimpleImputer → 가장 기본적인 결측치 대체 클래스
- Pandas의 .fillna()와 같은 기능 but 훈련/검증/배포 파이프라인에 자동으로 포함 가능, fit / transform 구조로 재현성 유지 가능

In [32]:
#### 용법 ####
# fit(X) : 데이터 X에서 채울 값(평균, 중앙값 등)을 계산

# transform(X) : 계산된 값으로 결측치를 실제로 채움

# fit_transform(X) : 두 과정을 한 번에 수행

In [33]:
from sklearn.impute import SimpleImputer

imputer_mean = SimpleImputer(strategy='mean') # strategy 안에 들어 갈 값: mean, median, most_frequent, constant(fill_value)
df[['국어','영어','수학','과학','역사']] = imputer_mean.fit_transform(df[['국어','영어','수학','과학','역사']]) # fit(열별 평균 계산) + transform(계산된 평균으로 NaN값 채우기) 한 번에 수행
df.isnull().sum()

이름    5
나이    8
성별    5
지역    3
국어    0
영어    0
수학    0
과학    0
역사    0
dtype: int64

In [34]:
# 성별, 지역 최빈값
imputer_most_frequent = SimpleImputer(strategy='most_frequent') # mean, median, most_frequent, constant(fill_value)
df[['성별', '지역']] = imputer_most_frequent.fit_transform(df[['성별', '지역']])

In [35]:
# 특정 열을 고정값으로 채우기 (예: 나이=18)
imputer_constant = SimpleImputer(strategy='constant',fill_value=18)
df['나이'] = imputer_constant.fit_transform(df[['나이']])
df.isnull().sum()

이름    5
나이    0
성별    0
지역    0
국어    0
영어    0
수학    0
과학    0
역사    0
dtype: int64

In [36]:
# 데이터 전처리
import random
data = {
    '이름': ['민준', '서준', '예준', '지우', '서윤', '도윤', '하준', '시우', '지후', '준우', '하윤'],
    '나이': [random.randint(18, 70) for _ in range(11)],
    '월급': [random.randint(2000, 8000) for _ in range(11)],
    '근무시간': [random.randint(20, 60) for _ in range(11)],
    '점수': [random.randint(50, 100) for _ in range(11)]
}
data['월급'][random.randint(0, 0)] = 20000 # 위 범위는 2000~8000인데, 20000으로 의도적으로 큰 이상치 삽입
data['점수'][random.randint(0, 3)] = 350 # 위 범위는 50~100인데, 350으로 의도적으로 큰 이상치 삽입
df_test = pd.DataFrame(data)
df_test.head()

,이름,나이,월급,근무시간,점수
0,민준,60,20000,27,54
1,서준,21,7569,25,350
2,예준,51,7860,38,59
3,지우,25,2673,23,72
4,서윤,66,3490,32,76


In [37]:
from sklearn.preprocessing  import MinMaxScaler

df_scaled = df_test.copy()
scaler = MinMaxScaler()
df_scaled[['월급','점수']] = scaler.fit_transform(df_scaled[['월급','점수']])
df_scaled

# 특정 열(월급, 점수)을 0~1 범위로 변환 → 값의 크기 차이를 줄이기 위해
# 즉, 스케일을 맞춰주는 것 -> 정규화 진행

,이름,나이,월급,근무시간,점수
0,민준,60,1.000000,27,0.010033
1,서준,21,0.282565,25,1.000000
2,예준,51,0.299359,38,0.026756
3,지우,25,0.000000,23,0.070234
4,서윤,66,0.047152,32,0.083612
5,도윤,62,0.293126,45,0.110368
6,하준,24,0.026202,52,0.147157
7,시우,67,0.240665,36,0.063545
8,지후,24,0.025740,44,0.000000
9,준우,30,0.172159,41,0.163880


In [38]:
from sklearn.preprocessing import StandardScaler

df_scaled2 = df_test.copy()
scaler = StandardScaler()
df_scaled2[['월급','점수']] = scaler.fit_transform(df_scaled2[['월급','점수']])
df_scaled2.head(1)

# Z-score를 표준으로 하여 변환
# 데이터를 평균 0, 표준편차 1로 변환 → 평균 중심화(Centering) + 분산 조정(Scaling) 즉, 각 변수의 단위나 크기에 상관없이 비교 가능하게 만듦

'''
# 방식	             값 범위	       계산 기준	    해석
MinMaxScaler	0 ~ 1	             min~max	 크기비율 유지
StandardScaler	평균 0, 표준편차 1	   평균/분산	정규분포 가정
'''

'\n# 방식\t             값 범위\t       계산 기준\t    해석\nMinMaxScaler\t0 ~ 1\t             min~max\t 크기비율 유지\nStandardScaler\t평균 0, 표준편차 1\t   평균/분산\t정규분포 가정\n'

In [39]:
'''
iqr
q1
q3
'''
outlier_data = [
    {'이름': '강호', '나이': 30, '월급': 25000, '근무시간': 40, '점수': 90},
    {'이름': '민서', '나이': 40, '월급': 3000, '근무시간': 50, '점수': 200},
    {'이름': '은성', '나이': 35, '월급': 10000, '근무시간': 35, '점수': 150}
]

df_with_outliers = pd.concat([df_test, pd.DataFrame(outlier_data)], ignore_index=True)
df_with_outliers


#정상 데이터에 이상치 3명을 인위적으로 추가해 이후 IQR 기반으로 탐지 실습

,이름,나이,월급,근무시간,점수
0,민준,60,20000,27,54
1,서준,21,7569,25,350
2,예준,51,7860,38,59
3,지우,25,2673,23,72
4,서윤,66,3490,32,76
5,도윤,62,7752,45,84
6,하준,24,3127,52,95
7,시우,67,6843,36,70
8,지후,24,3119,44,51
9,준우,30,5656,41,100


In [40]:
# 이상치 경계값(Outlier Boundary) 계산
q1, q3 = df_test['월급'].quantile([0.25, 0.75])
iqr = q3-q1
lower = q1-1.5*iqr
upper = q3+1.5*iqr

outliers = df_with_outliers[ (df_with_outliers['월급'] < lower) | (df_with_outliers['월급'] > upper)] # 월급이 “이상치 경계 바깥”에 있는 행만 추출, OR 조건 |
outliers

# IQR (InterQuartile Range) = Q3 - Q1 데이터의 중간 50% 범위
# 보통 Q1 - 1.5×IQR, Q3 + 1.5×IQR 밖의 값은 “이상치”로 간주 -> 이전에도 여러번 설명 드렸듯이 꼭 이 기준이 절대적인 것은 아님 데이터 셋에 따라서 상이함.

,이름,나이,월급,근무시간,점수
0,민준,60,20000,27,54
11,강호,30,25000,40,90


In [41]:
df_with_outliers[ (df_with_outliers['월급'] >= lower) & (df_with_outliers['월급'] <= upper)] # 월급이 “이상치 경계 바깥”에 있는 행만 추출, and 조건 &

,이름,나이,월급,근무시간,점수
1,서준,21,7569,25,350
2,예준,51,7860,38,59
3,지우,25,2673,23,72
4,서윤,66,3490,32,76
5,도윤,62,7752,45,84
6,하준,24,3127,52,95
7,시우,67,6843,36,70
8,지후,24,3119,44,51
9,준우,30,5656,41,100
10,하윤,54,6318,38,86


In [42]:
'''
범주형
명목 : 순서가 없는 범주형 예시) 색상, 혈액형
순서 : 순서가 있는 범주형 예시) 초중고대, 만족도, 메달
'''

'\n범주형\n명목 : 순서가 없는 범주형 예시) 색상, 혈액형\n순서 : 순서가 있는 범주형 예시) 초중고대, 만족도, 메달\n'

In [43]:
data = {
    '색상': ['빨강', '파랑', '노랑', '파랑', '빨강', '노랑'], # 명목
    '동물': ['개', '고양이', '새', '고양이', '개', '새'], # 명목
    '지역': ['서울', '부산', '대구', '서울', '대전', '부산'], # 명목
    '학교': ['초등학교', '고등학교', '중학교', '고등학교', '초등학교', '중학교'], # 순서
    '난이도': ['낮음', '높음', '중간', '높음', '낮음', '중간'], # 순서
    '등급': ['1등급', '2등급', '3등급', '2등급', '1등급', '3등급'] # 순서
}
df_categorical = pd.DataFrame(data)  # 딕셔너리의 데이터프레임 변환
df_categorical.dtypes

색상     object
동물     object
지역     object
학교     object
난이도    object
등급     object
dtype: object

In [44]:
# object 타입(문자형) 컬럼명 리스트 추출
df_categorical.select_dtypes(include=object).columns.tolist()

['색상', '동물', '지역', '학교', '난이도', '등급']

In [45]:
# 색상 열의 고유값 배열과 고유값과 그 갯수 확인
df_categorical['색상'].unique(), df_categorical['색상'].nunique()

(array(['빨강', '파랑', '노랑'], dtype=object), 3)

In [46]:
# one-hot encoding
from sklearn.preprocessing import OneHotEncoder

OHE = OneHotEncoder() # handle_unknown='ignore' 초록 0  , 'error' 
# 훈련 시 등장하지 않았던 새로운 범주가 테스트에 나오면 에러가 나므로, 실무에서는 handle_unknown='ignore' 옵션을 사용(디폴트값)

col_cat = ['색상'] # 인코딩할 열 목록 지정
OHE.fit(df_categorical[col_cat]) # fit 학습, 색상 열의 카테고리 목록을 학습(빨강/파랑/노랑)
df_ohe = OHE.transform(df_categorical[col_cat]) # transform 변환, 학습된 카테고리 기준으로 희소 행렬(sparse, 0과1로 구성) 생성
# fit_transform --- train data O, test data X

df_ohe = pd.DataFrame(df_ohe.toarray(), columns=OHE.get_feature_names_out(col_cat)) # 희소행렬을 .toarray()로 밀집 배열로 바꿔 DataFrame화
#  -> get_feature_names_out으로 열 이름(예: 색상_노랑, 색상_빨강, 색상_파랑) 부여, why? 사람이 읽기 쉽게, 또 다른 DataFrame과 병합/모델 입력하기 쉽게

df_ohe # ['빨강', '파랑', '노랑', '파랑', '빨강', '노랑']

,색상_노랑,색상_빨강,색상_파랑
0,0.0,1.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,0.0,1.0,0.0
5,1.0,0.0,0.0


In [47]:
# 더미화(dummy화), 더미 변수(dummy variable) 생성 ???
# 쉽게 말해서 문자(범주형) 데이터를 숫자(0과 1)로 바꾸는 걸 의미함
# 더미(dummy)는 ‘가짜의, 대리의’라는 뜻 -> 원래 데이터(문자)를 그대로 쓸 수 없으니, 그 정보를 대신(dummy) 표현하는 숫자 변수를 만든다는 의미에서 “더미 변수” 


#더미화의 원리

# 각 고유값(범주)마다 새로운 열을 하나씩 만든다

# 원래 행의 값이 해당 범주라면 1, 아니면 0을 채운다

# 이 과정을 One-Hot Encoding(원-핫 인코딩)라고 부른다

In [48]:
# get_dummies()
df_encoded = pd.get_dummies(df_categorical, columns=['색상'], drop_first=True)
df_encoded # drop_first=True: 첫 번째 카테고리(색상_노랑) 열을 제거하여 다중공선성 문제 방지

# 입력 형태: DataFrame, columns에 인코딩 대상 지정
# 출력 형태: df_categorical + 원-핫된 색상 더미열(첫 범주 삭제)

,동물,지역,학교,난이도,등급,색상_빨강,색상_파랑
0,개,서울,초등학교,낮음,1등급,True,False
1,고양이,부산,고등학교,높음,2등급,False,True
2,새,대구,중학교,중간,3등급,False,False
3,고양이,서울,고등학교,높음,2등급,False,True
4,개,대전,초등학교,낮음,1등급,True,False
5,새,부산,중학교,중간,3등급,False,False


In [49]:
# drop_first=True의 이유

# 더미 변수는 한 열을 제거해도 정보가 유지
# 예를 들어 위에서도 색상_노랑을 빼더라도 빨강=1 혹은 파랑=1이면 자동으로 노랑=0임을 알 수 있음

# 그래서 이렇게 1개를 제거(drop_first=True)하면 다중공선성(multicollinearity) 문제를 예방할 수 있음

In [50]:
# 다중공선성(multicollinearity)
# “서로 강하게 상관된(중복된) 변수들이 함께 들어 있어서, 모델이 어떤 변수가 진짜 영향을 주는지 구분하지 못하는 상태”

In [51]:
# Label Encoding

In [52]:
# label encoding
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_categorical['학교_encoder'] = label_encoder.fit_transform(df_categorical['학교'])
# ['초등학교', '고등학교', '중학교', '고등학교', '초등학교', '중학교']
df_categorical['난이도_encoder'] = label_encoder.fit_transform(df_categorical['난이도'])
df_categorical['등급_encoder'] = label_encoder.fit_transform(df_categorical['등급'])
df_categorical['학교_encoder'] # ['1등급', '2등급', '3등급', '2등급', '1등급', '3등급']

0    2
1    0
2    1
3    0
4    2
5    1
Name: 학교_encoder, dtype: int64

In [53]:
# ordinalencoder
from sklearn.preprocessing import OrdinalEncoder

custom_order = [['낮음','중간','높음']]
encoder = OrdinalEncoder(categories=custom_order)
df_categorical['난이도_인코딩'] = encoder.fit_transform(df_categorical[['난이도']]).astype(int)
df_categorical

,색상,동물,지역,학교,난이도,등급,학교_encoder,난이도_encoder,등급_encoder,난이도_인코딩
0,빨강,개,서울,초등학교,낮음,1등급,2,0,0,0
1,파랑,고양이,부산,고등학교,높음,2등급,0,1,1,2
2,노랑,새,대구,중학교,중간,3등급,1,2,2,1
3,파랑,고양이,서울,고등학교,높음,2등급,0,1,1,2
4,빨강,개,대전,초등학교,낮음,1등급,2,0,0,0
5,노랑,새,부산,중학교,중간,3등급,1,2,2,1


In [54]:
# 데이터 분할
'''
train : 모델 학습, 70~80%
validation : 훈련중 모델 성능 평가, 10~15% valid
test : 훈련이 완료된 후에 최종 성능 평가, 학습-검증 새로운 데이터, 모델 일반화 성능, 10~15%
'''
import seaborn as sns
from sklearn.model_selection import train_test_split

tips = sns.load_dataset('tips')
tips.head() # y = tip

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [55]:
y = tips['tip']
X = tips.drop(columns='tip')
X.head()

,total_bill,sex,smoker,day,time,size
0,16.99,Female,No,Sun,Dinner,2
1,10.34,Male,No,Sun,Dinner,3
2,21.01,Male,No,Sun,Dinner,3
3,23.68,Male,No,Sun,Dinner,2
4,24.59,Female,No,Sun,Dinner,4


In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape # (행, 열)

((170, 6), (74, 6), (170,), (74,))

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [58]:
X.head()

,total_bill,sex,smoker,day,time,size
0,16.99,Female,No,Sun,Dinner,2
1,10.34,Male,No,Sun,Dinner,3
2,21.01,Male,No,Sun,Dinner,3
3,23.68,Male,No,Sun,Dinner,2
4,24.59,Female,No,Sun,Dinner,4


In [59]:
X_encoded = pd.get_dummies(X, columns=['sex','smoker','time'], drop_first=True)
X_encoded.head()

,total_bill,day,size,sex_Female,smoker_No,time_Dinner
0,16.99,Sun,2,True,True,True
1,10.34,Sun,3,False,True,True
2,21.01,Sun,3,False,True,True
3,23.68,Sun,2,False,True,True
4,24.59,Sun,4,True,True,True


In [60]:
X_encoded.shape

(244, 6)

In [61]:
from sklearn.preprocessing import OrdinalEncoder

X_enoded2 = X_encoded.copy()
custom = [['Sun', 'Sat', 'Fri', 'Thur']]
oe = OrdinalEncoder(categories=custom)
X_enoded2['day'] = oe.fit_transform(X_encoded[['day']]).astype(int)

In [62]:
X_enoded2.head()

,total_bill,day,size,sex_Female,smoker_No,time_Dinner
0,16.99,0,2,True,True,True
1,10.34,0,3,False,True,True
2,21.01,0,3,False,True,True
3,23.68,0,2,False,True,True
4,24.59,0,4,True,True,True


In [63]:
titanic = sns.load_dataset('titanic')
titanic.head() # y = survived

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [64]:
titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [65]:
titanic.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [66]:
titanic.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [67]:
titanic.dtypes

survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object

In [68]:
titanic['embarked'].fillna(titanic['embarked'].mode()[0], inplace=True)
titanic.isnull().sum()

C:\Users\lg\AppData\Local\Temp\ipykernel_2844\4240078869.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  titanic['embarked'].fillna(titanic['embarked'].mode()[0], inplace=True)


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [69]:
# age 결측치 평균값 채우기
titanic['age'].fillna(titanic['age'].mean(), inplace=True)

C:\Users\lg\AppData\Local\Temp\ipykernel_2844\990191272.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  titanic['age'].fillna(titanic['age'].mean(), inplace=True)


In [70]:
titanic.isnull().sum()

survived         0
pclass           0
sex              0
age              0
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [71]:
titanic.drop('deck', axis=1, inplace=True)

In [72]:
titanic.fillna(
    {
        'embark_town' : titanic['embark_town'].mode()[0]
    }
    ,inplace=True
)

In [73]:
titanic.isna().sum() #NaN

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [74]:
titanic['has_family'] = (titanic['sibsp'] + titanic['parch']).apply(lambda x: 1 if x > 0 else 0)
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,has_family
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False,1
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False,1
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True,0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False,1
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,0


In [75]:
titanic['age'].nunique()

89

In [76]:
titanic['age'].unique()

array([22.        , 38.        , 26.        , 35.        , 29.69911765,
       54.        ,  2.        , 27.        , 14.        ,  4.        ,
       58.        , 20.        , 39.        , 55.        , 31.        ,
       34.        , 15.        , 28.        ,  8.        , 19.        ,
       40.        , 66.        , 42.        , 21.        , 18.        ,
        3.        ,  7.        , 49.        , 29.        , 65.        ,
       28.5       ,  5.        , 11.        , 45.        , 17.        ,
       32.        , 16.        , 25.        ,  0.83      , 30.        ,
       33.        , 23.        , 24.        , 46.        , 59.        ,
       71.        , 37.        , 47.        , 14.5       , 70.5       ,
       32.5       , 12.        ,  9.        , 36.5       , 51.        ,
       55.5       , 40.5       , 44.        ,  1.        , 61.        ,
       56.        , 50.        , 36.        , 45.5       , 20.5       ,
       62.        , 41.        , 52.        , 63.        , 23.5 

In [77]:
bins= [0, 10, 20, 30, 40, 50, 60, 70, 80] # 왼쪽 포함 X, 오른쪽 포함
labels = ['child','teenager', 'young adult','adult','middle aged','senior','elder','oldest']
titanic['age_group'] = pd.cut(titanic['age'],bins=bins, labels=labels)
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,has_family,age_group
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False,1,young adult
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False,1,adult
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True,0,young adult
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False,1,adult
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,0,adult


In [78]:
def detect_outliers(df, column):
  q1 = df[column].quantile(0.25)
  q3 = df[column].quantile(0.75)

  iqr = q3-q1

  lower = q1-1.5*iqr
  upper = q3+1.5*iqr

  outliers = df[ (df[column] < lower) | (df[column] > upper)]
  print(lower, upper)
  print(len(outliers))
  df_clean = df[ (df[column] >= lower)  & (df[column] <= upper) ]
  return df_clean

titanic = detect_outliers(titanic, 'age')
titanic = detect_outliers(titanic, 'fare')
titanic.head()

2.5 54.5
66
-25.366699999999994 63.333299999999994
107


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,has_family,age_group
0,0,3,male,22.000000,1,0,7.2500,S,Third,man,True,Southampton,no,False,1,young adult
2,1,3,female,26.000000,0,0,7.9250,S,Third,woman,False,Southampton,yes,True,0,young adult
3,1,1,female,35.000000,1,0,53.1000,S,First,woman,False,Southampton,yes,False,1,adult
4,0,3,male,35.000000,0,0,8.0500,S,Third,man,True,Southampton,no,True,0,adult
5,0,3,male,29.699118,0,0,8.4583,Q,Third,man,True,Queenstown,no,True,0,young adult


In [79]:
upper_limit = titanic['fare'].quantile(0.99)
titanic['fare'] = titanic['fare'].apply(lambda x: upper_limit if x > upper_limit else x)

In [80]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
titanic['age'] = scaler.fit_transform(titanic[['age']])

In [81]:
titanic.groupby('survived')[['age','fare','sibsp','parch']].mean()

,age,fare,sibsp,parch
survived,,,,
0,0.047860,14.853703,0.412134,0.278243
1,-0.095321,21.796842,0.416667,0.395833


In [82]:
obj_cols = titanic.select_dtypes(include='object')
for col in obj_cols:
  print(col, titanic[col].unique())

sex ['male' 'female']
embarked ['S' 'Q' 'C']
who ['man' 'woman' 'child']
embark_town ['Southampton' 'Queenstown' 'Cherbourg']
alive ['no' 'yes']


In [83]:
# 범주형
titanic = pd.get_dummies(titanic, columns=['embarked'], drop_first=True)
titanic.head(1)

,survived,pclass,sex,age,sibsp,parch,fare,class,who,adult_male,embark_town,alive,alone,has_family,age_group,embarked_Q,embarked_S
0,0,3,male,-0.644501,1,0,7.25,Third,man,True,Southampton,no,False,1,young adult,False,True


In [84]:
titanic['sex'] = titanic['sex'].map({'male':0, 'female':1})
titanic.head(1)

,survived,pclass,sex,age,sibsp,parch,fare,class,who,adult_male,embark_town,alive,alone,has_family,age_group,embarked_Q,embarked_S
0,0,3,0,-0.644501,1,0,7.25,Third,man,True,Southampton,no,False,1,young adult,False,True


In [85]:
obj_cols = titanic.select_dtypes(include='object').columns.tolist()
obj_cols

['who', 'embark_town', 'alive']

In [86]:
# label encoder 남은 범주형 데이터 바꿔보세요.
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
for col in obj_cols:
  titanic[col] = label_encoder.fit_transform(titanic[[col]])

titanic.head(1)

c:\Users\lg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\lg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\lg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,survived,pclass,sex,age,sibsp,parch,fare,class,who,adult_male,embark_town,alive,alone,has_family,age_group,embarked_Q,embarked_S
0,0,3,0,-0.644501,1,0,7.25,Third,1,True,2,0,False,1,young adult,False,True


In [88]:
from sklearn.model_selection import train_test_split

X = titanic.drop('survived', axis=1)
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(X, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

ValueError: not enough values to unpack (expected 4, got 2)